# 基礎編18

V3.7新機能:　コントラクトAPI:　isReadMode()

このノートブックでは、スマートコントラクト・バージョン3.7から利用できるコントラクトAPIであるisReadMode()について説明します。    

## isReadMode()の機能の説明

isReadMode()は、現在実行中のスマートコントラクトがリードモードで動作しているかどうかを真偽値で返します。

## 準備

In [25]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

## スマートコントラクトのデプロイ

isReadMode()を使った簡単なスマートコントラクトをデプロイします。  
スマートコントラクトの処理の最初で、リードモードかどうかをチェックし、リードモードでなければ例外をスローします。  
これにより、このスマートコントラクトは実質的にリードモードでしか実行できないものとなります。  

In [26]:
var cid = await deploySmartContract(function basic18(){
    if(isReadMode() != true) throw 'リードモードで実行してください';
    // リードモードで実行する処理
    return '実行しました';
});

ドメイン内の別のコントラクトからサブコントラクトとして呼び出せるように、実行権をドメイン内に開放します。

In [27]:
var resp = await rpc.call(adminWallet, 'c1update', {id: cid, prop: 'accessible_to', value: [ domain ]});
console.log(resp);

{
  txno: 207290,
  txid: 'xrFfvoZ9wvVFpbhAoJrTrujkxybYSfJqnHf7HUrwZPpDQB',
  status: 'ok',
  value: null
}


## リードモードで実行する場合

In [28]:
var resp = await rpc.call(adminWallet, cid, {}, { readmode: 'fast' });
console.log(resp);

{
  txno: 207291,
  txid: 'xTJS5AtUkYNuc3CGtBppANo92sKccgNaEbRL6nafd9KxGB',
  status: 'read',
  value: '実行しました'
}


readmode: 'full'で実行する場合も同様です。

In [29]:
var resp = await rpc.call(adminWallet, cid, {}, { readmode: 'full' });
console.log(resp);

{
  txno: 207291,
  txid: 'xrib8aagaiR2qQBwR9FrVuEyfaaFe4D7RGtyfuty4wn6u',
  status: 'read',
  value: '実行しました'
}


## リードモードでない実行の場合

In [30]:
var resp = await rpc.call(adminWallet, cid, {});
console.log(resp);

{
  txno: 207292,
  txid: 'x6zJBr3pEH5Z5QdeYLxUCmUUUFSCj4yUcyvEzLFqVH3vq',
  status: 'thrown',
  value: 'リードモードで実行してください'
}


この場合、例外が発生し、トランザクションはエラー終了します。

## サブコントラクトの場合

上で作ったスマートコントラクトをサブコントラクトとして呼び出すスマートコントラクトをデプロイします。(__subcid__はサブコントラクトのIDに置換されます)  
サブコントラクト呼び出しは、リードモードを指定します。（コントラクト内では真偽値で指定します。）

In [31]:
var cid1 = await deploySmartContract(function basic18p1(){
    return openContract('__subcid__').call({}, {readmode: true});
}, {
    replacers: [['__subcid__', cid ]]
});

この場合、親コントラクトをリードモードで呼び出したかどうかに関わらず、サブコントラクトは正常に完了します。

In [32]:
var resp = await rpc.call(adminWallet, cid1, {}, { readmode: 'fast' });
console.log(resp);

{
  txno: 207297,
  txid: 'x8VMUkc3JRJHDNiP59SHkQ2QFiTVQsrtG67wdTHYt8T6NB',
  status: 'read',
  value: '実行しました'
}


上の場合、サブコントラクト呼び出し時にreadmodeを指定しているため、サブコントラクト内ではisReadMode()がtrueとなり正常終了します。

In [33]:
var resp = await rpc.call(adminWallet, cid1, {});
console.log(resp);

{
  txno: 207297,
  txid: 'xhpyR88qJJU75iXJ4pdX7RwY9jERvAp4vHuYfgn4MLoev',
  status: 'ok',
  value: '実行しました'
}


上で作ったスマートコントラクトをサブコントラクトとして呼び出す別のスマートコントラクトをデプロイします。  (__subcid__はサブコントラクトのIDに置換されます)  

今度は、サブコントラクト呼び出しでリードモードを指定しません。

In [34]:
var cid2 = await deploySmartContract(function basic18p2(){
    return openContract('__subcid__').call();
}, {
    replacers: [['__subcid__', cid ]]
});

この場合、親コントラクトをリードモードで呼び出したときに限り、サブコントラクトは正常に完了します。

In [35]:
var resp = await rpc.call(adminWallet, cid2, {}, { readmode: 'fast' });
console.log(resp);

{
  txno: 207303,
  txid: 'xbBpdHmVBbHhVLsvqemRDY9bRX8QGbnY8xtUY7fPKhWop',
  status: 'read',
  value: '実行しました'
}


今度のケースではサブコントラクトの呼び出し時にreadmodeを指定しないため、親がリードモードでないとサブコントラクト内のisReadMode()がfalseとなりエラーになります。

In [36]:
var resp = await rpc.call(adminWallet, cid2, {});
console.log(resp);

{
  txno: 207303,
  txid: 'xYMwyToaBEGi43FdjnxKMG6gYPzZRokxHVFEeLus5395HB',
  status: 'aborted',
  value: 'thrown: リードモードで実行してください\n    at c076221508:1:12'
}
